In [1]:
import pandas as pd
import re
from collections import Counter

In [3]:
file_path = r'D:\TTS_ITMO\tts_labs_itmo\labs\lab1_text\data\metadata_RUSLAN_22200.csv'

In [4]:
with open(file_path, 'r', encoding='utf-8') as f:
    for i in range(10):
        line = f.readline()
        print(f"Строка {i+1}: {repr(line)}")

Строка 1: '000000_RUSLAN|С тревожным чувством берусь я за перо.\n'
Строка 2: '000001_RUSLAN|Кого интересуют признания литературного неудачника?\n'
Строка 3: '000002_RUSLAN|Что поучительного в его исповеди?\n'
Строка 4: '000003_RUSLAN|Да и жизнь моя лишена внешнего трагизма.\n'
Строка 5: '000004_RUSLAN|Я абсолютно здоров.\n'
Строка 6: '000005_RUSLAN|У меня есть любящая родня.\n'
Строка 7: '000006_RUSLAN|Мне всегда готовы предоставить работу, которая обеспечит нормальное биологическое существование.\n'
Строка 8: '000007_RUSLAN|Мало того, я обладаю преимуществами.\n'
Строка 9: '000008_RUSLAN|Мне без труда удается располагать к себе людей.\n'
Строка 10: '000009_RUSLAN|Я совершил десятки поступков, уголовно наказуемых и оставшихся безнаказанными.\n'


In [5]:
texts = pd.read_csv(file_path, encoding = 'utf-8', sep = '|', header=None, names = ['filename', 'text'])

In [6]:
texts.head(10)

,filename,text
0,000000_RUSLAN,С тревожным чувством берусь я за перо.
1,000001_RUSLAN,Кого интересуют признания литературного неудач...
2,000002_RUSLAN,Что поучительного в его исповеди?
3,000003_RUSLAN,Да и жизнь моя лишена внешнего трагизма.
4,000004_RUSLAN,Я абсолютно здоров.
5,000005_RUSLAN,У меня есть любящая родня.
6,000006_RUSLAN,"Мне всегда готовы предоставить работу, которая..."
7,000007_RUSLAN,"Мало того, я обладаю преимуществами."
8,000008_RUSLAN,Мне без труда удается располагать к себе людей.
9,000009_RUSLAN,"Я совершил десятки поступков, уголовно наказуе..."


In [7]:
def has_non_standart_chars(text):
    allowed = r'[^а-яА-ЯёЁ\s\.,!?-]'
    return bool(re.search(allowed, text))

def find_bad_chars(text):
    bad_pattern = r'[^а-яА-ЯёЁ\s\.,!?-]'
    return re.findall(bad_pattern, text)

In [8]:
texts['has_bad_chars'] = texts['text'].apply(has_non_standart_chars)
texts['bad_chars'] = texts['text'].apply(find_bad_chars)

In [9]:
problematic = texts[texts['has_bad_chars']]

In [10]:
problematic

,filename,text,has_bad_chars,bad_chars
583,000583_RUSLAN,"Честен, принципиален, морально устойчив…",True,[…]
845,000845_RUSLAN,Мечтаем получить от вас рецензию.”,True,[”]
880,000880_RUSLAN,Я на букву О /библиография к Окуджаве/.,True,"[/, /]"
1024,001024_RUSLAN,"А Лев Уфлянд* еще больше подливает желчи, плюе...",True,[*]
1910,001910_RUSLAN,"Человеку, которым занимаются соответствующие о...",True,[…]
...,...,...,...,...
22185,022185_RUSLAN,"Дальше – газета, радиостанция «Либерти»… Одна ...",True,"[–, «, », …, –, …]"
22186,022186_RUSLAN,Развлечение у меня единственное – сигареты. Я ...,True,"[–, …]"
22189,022189_RUSLAN,"Помните, вы говорили «Сережу мысли не интересу...",True,"[«, …, »]"
22190,022190_RUSLAN,(Я представляю себе вашу ироническую улыбку. Т...,True,"[(, –, )]"


In [12]:
texts['text']

0                   С тревожным чувством берусь я за перо.
1        Кого интересуют признания литературного неудач...
2                        Что поучительного в его исповеди?
3                 Да и жизнь моя лишена внешнего трагизма.
4                                      Я абсолютно здоров.
                               ...                        
22195    Мы жаждем совершенства, а вокруг торжествует п...
22196    Революционер делает попытки установить мировую...
22197    Он начинает преобразовывать жизнь, достигая ин...
22198    Допустим, выводит морковь, совершенно неотличи...
22199    Известно, чем это кончается… Что в этой ситуац...
Name: text, Length: 22200, dtype: object

Примеры слов, найденных как 'аббревиатуры':


In [31]:
#Латиница
texts['has_latin'] = texts['text'].str.contains(r'[a-zA-Z]', regex=True)
latin_rows = texts[texts['has_latin']]

latin_count = len(latin_rows)
latin_pct = latin_count / len(texts) * 100

print(f"✅ Найдено строк с латиницей: {latin_count} ({latin_pct:.2f}%)")

if latin_count > 0:
    print("\n📌 Примеры строк с латиницей (первые 10):")
    for idx, row in latin_rows.head(10).iterrows():
        found = re.findall(r'[a-zA-Z]+', row['text'])
        print(f"\n  {idx}. {row['filename']}")
        print(f"     Найдено: {found}")
        print(f"     Текст: {row['text'][:100]}...")
else:
    print("\n   ❌ Латиница не найдена в корпусе!")

✅ Найдено строк с латиницей: 0 (0.00%)

   ❌ Латиница не найдена в корпусе!


In [34]:
#цифры
texts['has_digits'] = texts['text'].str.contains(r'\d', regex=True)
digit_rows = texts[texts['has_digits']]

digit_count = len(digit_rows)
digit_pct = digit_count / len(texts) * 100

print(f"Всего строк в корпусе: {len(texts)}")
print(f"✅ Найдено строк с цифрами: {digit_count} ({digit_pct:.2f}%)")

if digit_count > 0:
    print("\n📌 Примеры строк с цифрами (первые 10):")
    for idx, row in digit_rows.head(10).iterrows():
        found = re.findall(r'\d+', row['text'])
        print(f"\n  {idx}. {row['filename']}")
        print(f"     Найдено: {found}")
        print(f"     Текст: {row['text'][:100]}...")
else:
    print("\n   ❌ Цифры не найдены в корпусе!")


Всего строк в корпусе: 22200
✅ Найдено строк с цифрами: 4 (0.02%)

📌 Примеры строк с цифрами (первые 10):

  9017. 009017_RUSLAN
     Найдено: ['61']
     Текст: Брат разъезжал по отдаленным лагерным точкам. Ему предоставили казенную машину «ГАЗ‑61». ...

  11906. 011906_RUSLAN
     Найдено: ['129']
     Текст: На Филиппинах кто‑то застрелил руководителя партийной оппозиции. Под Мелитополем разбился «ТУ‑129». ...

  16720. 016720_RUSLAN
     Найдено: ['16']
     Текст: В сумочке ее лежало нечто, размером чуть поболее миниатюрного дамского браунинга «Элита‑16». ...

  21885. 021885_RUSLAN
     Найдено: ['6']
     Текст: – Турбовинтовой МИ‑6, – заметил Пупс, вставая. – Е‑ё! – лениво крикнул он. Затем скрестил над голово...


In [35]:
#техсимволы
texts['has_tech'] = texts['text'].str.contains(r'[*\/@+<>{}[\]()=#~`]', regex=True)
tech_rows = texts[texts['has_tech']]

tech_count = len(tech_rows)
tech_pct = tech_count / len(texts) * 100

print(f"Всего строк в корпусе: {len(texts)}")
print(f"✅ Найдено строк с техническими символами: {tech_count} ({tech_pct:.2f}%)")

if tech_count > 0:
    print("\n📌 Примеры строк с техническими символами (первые 10):")
    for idx, row in tech_rows.head(10).iterrows():
        found = re.findall(r'[*\/@+<>{}[\]()=#~`]', row['text'])
        print(f"\n  {idx}. {row['filename']}")
        print(f"     Найдено: {found}")
        print(f"     Текст: {row['text'][:100]}...")
else:
    print("\n   ❌ Технические символы не найдены в корпусе!")

Всего строк в корпусе: 22200
✅ Найдено строк с техническими символами: 312 (1.41%)

📌 Примеры строк с техническими символами (первые 10):

  880. 000880_RUSLAN
     Найдено: ['/', '/']
     Текст: Я на букву О /библиография к Окуджаве/....

  1024. 001024_RUSLAN
     Найдено: ['*']
     Текст: А Лев Уфлянд* еще больше подливает желчи, плюет на русский народ....

  4530. 004530_RUSLAN
     Найдено: ['/']
     Текст: /Например, вертухай, как вы соизволили дружески меня поименовать....

  5814. 005814_RUSLAN
     Найдено: ['(', ')']
     Текст: «СОПЕРНИКИ ВЕТРА (Таллиннскому ипподрому пятьдесят лет). Известные жокеи. ...

  5846. 005846_RUSLAN
     Найдено: ['(', ')']
     Текст: Дукель (то есть Дукальский) ставит через приезжих латышей. Это крутой солидняк. ...

  5853. 005853_RUSLAN
     Найдено: ['(', ')']
     Текст: Компромисс третий «Я ЧУВСТВУЮ СЕБЯ КАК ДОМА (Гости Таллинна). ...

  5893. 005893_RUSLAN
     Найдено: ['(', ')']
     Текст: Один раз грузчики товарной станции (им это у

In [36]:
#emoji
emoji_pattern = re.compile("["
    u"\U0001F600-\U0001F64F"  
    u"\U0001F300-\U0001F5FF" 
    u"\U0001F680-\U0001F6FF"
    u"\U00002702-\U000027B0"
    u"\U000024C2-\U0001F251" 
    "]+", flags=re.UNICODE)
texts['has_emoji'] = texts['text'].apply(lambda x: bool(emoji_pattern.search(x)))
emoji_rows = texts[texts['has_emoji']]

emoji_count = len(emoji_rows)
emoji_pct = emoji_count / len(texts) * 100

print(f"Всего строк в корпусе: {len(texts)}")
print(f"✅ Найдено строк с эмодзи: {emoji_count} ({emoji_pct:.2f}%)")

if emoji_count > 0:
    print("\n📌 Примеры строк с эмодзи (первые 10):")
    for idx, row in emoji_rows.head(10).iterrows():
        found = emoji_pattern.findall(row['text'])
        print(f"\n  {idx}. {row['filename']}")
        print(f"     Найдено: {found}")
        print(f"     Текст: {row['text'][:100]}...")
else:
    print("\n   ❌ Эмодзи не найдены в корпусе!")

Всего строк в корпусе: 22200
✅ Найдено строк с эмодзи: 0 (0.00%)

   ❌ Эмодзи не найдены в корпусе!


In [1]:
#shortens
stop_words = [
    "он", "ты", "мы", "вы", "я", "она", "они", "то", "все", "его", "ее", "же", "бы", "да", "их", "ей", "уж", "ну",
    "мне", "те", "се", "вас", "нас", "сам", "ах", "ох", "эх", "ой", "ай", "эт", "ли",
    "меня", "тебя", "себя", "вами", "нами", "мой", "твой", "свой",
    "это", "эти", "эта", "тот", "та", "те", "вон", "вот",
    "кто", "что", "чей", "чья", "чьё",
    "весь", "вся", "всё", "сама", "сами",
    "ага", "увы", "ого"
]
def find_abbr_with_filter(text):
    candidates = re.findall(r'\b[а-яА-Я]{1,2}\.', text)
    
    # Фильтруем: исключаем стоп-слова
    real_abbr = []
    for word in candidates:
        word_without_dot = word[:-1].lower()
        if word_without_dot not in stop_words:
            real_abbr.append(word)
    
    return real_abbr

texts['abbr_found'] = texts['text'].apply(find_abbr_with_filter)
texts['has_abbr'] = texts['abbr_found'].apply(lambda x: len(x) > 0)
abbr_rows = texts[texts['has_abbr']]

abbr_count = len(abbr_rows)
abbr_pct = abbr_count / len(texts) * 100

print(f"Всего строк в корпусе: {len(texts)}")
print(f"✅ Найдено строк с сокращениями: {abbr_count} ({abbr_pct:.2f}%)")

# Собираем все найденные сокращения для статистики
all_abbr = []
texts['abbr_found'].apply(lambda x: all_abbr.extend(x))
from collections import Counter
abbr_counts = Counter(all_abbr)

print(f"\n📊 Найденные сокращения и их частота:")
if abbr_counts:
    for abbr, count in abbr_counts.most_common(20):
        print(f"     {abbr}: {count} раз")
else:
    print("     ❌ Сокращения не найдены")

if abbr_count > 0:
    print("\n📌 Примеры строк с сокращениями (первые 10):")
    for idx, row in abbr_rows.head(10).iterrows():
        print(f"\n  {idx}. {row['filename']}")
        print(f"     Найдено: {row['abbr_found']}")
        print(f"     Текст: {row['text'][:100]}...")
        
        # Проверяем, какие слова были исключены стоп-списком
        all_candidates = re.findall(r'\b[а-яА-Я]{1,2}\.', row['text'])
        filtered_out = [w for w in all_candidates if w[:-1].lower() in stop_words]
        if filtered_out:
            print(f"     Исключено (стоп-слова): {filtered_out}")
else:
    print("\n   ❌ Сокращения не найдены в корпусе!")

NameError: name 'texts' is not defined

In [51]:
#аббревиатуры (без гласных)
def find_true_acronyms(text):
    words = re.findall(r'\b[А-ЯЁ]{2,}\b', text)
    vowels = 'АЕЁИОУЫЭЮЯ'
    return [w for w in words if not any(v in w for v in vowels)]

texts['true_acronyms'] = texts['text'].apply(find_true_acronyms)
texts['has_true_acronym'] = texts['true_acronyms'].apply(lambda x: len(x) > 0)



print(f"Строк с аббревиатурами: {texts['has_true_acronym'].sum()}")
print(f"Это {texts['has_true_acronym'].sum() / len(texts) * 100:.2f}% от всех строк")

# Собираем все настоящие аббревиатуры
all_true_acronyms = []
texts['true_acronyms'].apply(lambda x: all_true_acronyms.extend(x))
true_acronym_counts = Counter(all_true_acronyms)

print(f"\nНастоящие аббревиатуры:")
if true_acronym_counts:
    for acr, count in true_acronym_counts.most_common():
        print(f"  {acr}: {count} раз")
else:
    print("  ❌ Настоящие аббревиатуры не найдены!")



Строк с аббревиатурами: 31
Это 0.14% от всех строк

Настоящие аббревиатуры:
  КПСС: 6 раз
  НКВД: 4 раз
  ЦДЛ: 4 раз
  МТС: 2 раз
  ВМК: 2 раз
  ЧК: 2 раз
  МВД: 2 раз
  КП: 1 раз
  БТ: 1 раз
  СС: 1 раз
  ССР: 1 раз
  КВН: 1 раз
  КВВК: 1 раз
  КПП: 1 раз
  ФБР: 1 раз
  СХШ: 1 раз
  ГБ: 1 раз
  ФД: 1 раз


In [52]:
def is_clean(text):
    pattern = r'^[а-яА-ЯёЁ\s\.,!?-]+$'
    return bool(re.match(pattern, text))

In [53]:
texts['is_clean'] = texts['text'].apply(is_clean)
clean_rows = texts[texts['is_clean']]

In [54]:
for text in clean_rows['text'].head(5):
    print(f" {text}")

 С тревожным чувством берусь я за перо.
 Кого интересуют признания литературного неудачника?
 Что поучительного в его исповеди?
 Да и жизнь моя лишена внешнего трагизма.
 Я абсолютно здоров.


In [60]:


# Для настоящих аббревиатур (без гласных)
true_acro_rows = texts[texts['has_true_acronym']]  # 



total = len(texts)

# Настоящие аббревиатуры (без гласных)
true_acro_count = texts['has_true_acronym'].sum()
true_acro_pct = true_acro_count / total * 100

# Все слова заглавными (включая имена)
all_caps_count = texts['has_acronym'].sum()
all_caps_pct = all_caps_count / total * 100

print(f"Всего строк: {total}")

print(f"\n2. аббревиатуры (без гласных):")
print(f"   {true_acro_count} строк ({true_acro_pct:.2f}%)")


if true_acro_count > 0:
    # Показываем настоящие аббревиатуры
    all_true_acr = []
    texts['true_acronyms'].apply(lambda x: all_true_acr.extend(x))
    from collections import Counter
    acr_counts = Counter(all_true_acr)
    
    print(f"\n📌 Настоящие аббревиатуры в корпусе:")
    for acr, count in acr_counts.most_common(20):
        print(f"  {acr}: {count} раз")
else:
    print("\n   ❌ Настоящие аббревиатуры не найдены!")

Всего строк: 22200

2. аббревиатуры (без гласных):
   31 строк (0.14%)

📌 Настоящие аббревиатуры в корпусе:
  КПСС: 6 раз
  НКВД: 4 раз
  ЦДЛ: 4 раз
  МТС: 2 раз
  ВМК: 2 раз
  ЧК: 2 раз
  МВД: 2 раз
  КП: 1 раз
  БТ: 1 раз
  СС: 1 раз
  ССР: 1 раз
  КВН: 1 раз
  КВВК: 1 раз
  КПП: 1 раз
  ФБР: 1 раз
  СХШ: 1 раз
  ГБ: 1 раз
  ФД: 1 раз


In [64]:

total = len(texts)

# 1. Чистые строки
clean_rows = texts[~texts['has_bad_chars']]
print(f"\nЧистые строки: {len(clean_rows)} ({len(clean_rows)/total*100:.1f}%)")

# 2. Проблемные строки
problematic = texts[texts['has_bad_chars']]
print(f"Проблемные строки: {len(problematic)} ({len(problematic)/total*100:.1f}%)")

# 3. Сокращения (ПРАВИЛЬНО!)
abbr_rows = texts[texts['has_abbr']]
print(f"\nСокращения (ул., г., пр., т.д.): {len(abbr_rows)} ({len(abbr_rows)/total*100:.2f}%)")

# 4. Настоящие аббревиатуры (ПРАВИЛЬНО!)
true_acro_rows = texts[texts['has_true_acronym']]
print(f"Настоящие аббревиатуры (без гласных): {len(true_acro_rows)} ({len(true_acro_rows)/total*100:.2f}%)")

# 5. Все заглавные слова (НЕ аббревиатуры!)
all_caps_rows = texts[texts['has_acronym']]
print(f"\nВсе слова заглавными (имена, названия): {len(all_caps_rows)} ({len(all_caps_rows)/total*100:.1f}%)")
print(f" Это имена собственные и названия")


Чистые строки: 12953 (58.3%)
Проблемные строки: 9247 (41.7%)

Сокращения (ул., г., пр., т.д.): 101 (0.45%)
Настоящие аббревиатуры (без гласных): 31 (0.14%)

Все слова заглавными (имена, названия): 276 (1.2%)
 Это имена собственные и названия


In [69]:


problem_types = [
    ("Технические символы", 'has_tech'),
    ("Аббревиатуры (все заглавные)", 'has_acronym'),
    ("Сокращения", 'has_abbr'),
    ("Аббревиатуры (без гласных)", 'has_true_acronym'),
    ("Цифры", 'has_digits'),
    ("Латиница", 'has_latin'),
    ("Эмодзи", 'has_emoji'),
    ("Кавычки", 'has_quotes'),
    ("Тире", 'has_dashes'),
    ("Многоточие", 'has_ellipsis'),
]

total = len(texts)

for name, col in problem_types:
    count = texts[col].sum() if col in texts.columns else 0
    if count > 0:
        print(f"  • {name:30} {count:>6} ({count/total*100:>5.1f}%)")
    else:
        print(f"  • {name:30} {count:>6} ({0.0:>5.1f}%)")

  • Технические символы               312 (  1.4%)
  • Аббревиатуры (все заглавные)      276 (  1.2%)
  • Сокращения                        101 (  0.5%)
  • Аббревиатуры (без гласных)         31 (  0.1%)
  • Цифры                               4 (  0.0%)
  • Латиница                            0 (  0.0%)
  • Эмодзи                              0 (  0.0%)
  • Кавычки                             0 (  0.0%)
  • Тире                                0 (  0.0%)
  • Многоточие                          0 (  0.0%)


In [61]:
# Собираем все проблемные символы
all_bad_chars = []
texts['bad_chars'].apply(lambda x: all_bad_chars.extend(x))

char_counts = Counter(all_bad_chars)

print("\n" + "=" * 80)
print("ТОП САМЫХ ЧАСТЫХ ПРОБЛЕМНЫХ СИМВОЛОВ")
print("=" * 80)

for char, count in char_counts.most_common(30):
    # Показываем символ и его код для наглядности
    try:
        char_name = repr(char)  # Показывает символ с кавычками
        print(f"  {char_name} : {count} раз")
    except:
        print(f"  '{char}' : {count} раз")
all_unique_bad = sorted(set(all_bad_chars))
print(f"\nВсего уникальных проблемных символов: {len(all_unique_bad)}")
print("Список всех уникальных проблемных символов:")
print(all_unique_bad)


ТОП САМЫХ ЧАСТЫХ ПРОБЛЕМНЫХ СИМВОЛОВ
  '–' : 10410 раз
  '…' : 4097 раз
  '‑' : 2243 раз
  '«' : 1999 раз
  '»' : 1997 раз
  '(' : 305 раз
  ')' : 305 раз
  '„' : 72 раз
  '“' : 57 раз
  '”' : 13 раз
  '’' : 6 раз
  ';' : 5 раз
  '/' : 4 раз
  '́' : 4 раз
  '6' : 3 раз
  '1' : 3 раз
  '*' : 1 раз
  '2' : 1 раз
  '9' : 1 раз
  '<' : 1 раз
  '>' : 1 раз

Всего уникальных проблемных символов: 21
Список всех уникальных проблемных символов:
['(', ')', '*', '/', '1', '2', '6', '9', ';', '<', '>', '«', '»', '́', '‑', '–', '’', '“', '”', '„', '…']
